In [23]:
!pip install black==24.3.0 --no-deps
!pip install pylint==2.17.7 --no-deps

# Configuracion

In [ ]:
team = 'RIESGOS'  # RETAIL, RIESGOS
name_ds = ''  # TODO 2: Colocar mis apellidos y nombres
path_script = './inference.py'


In [25]:
import os
import boto3
from sagemaker import get_execution_role

account = boto3.client('sts').get_caller_identity()['Account']
image_uri = f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.8.15-cpu'
role_arn = get_execution_role()

os.environ['I_TEAM_RETAIL'], os.environ['I_CC_RETAIL'] = 'DS RETAIL', '9946100000'
os.environ['I_TEAM_RIESGOS'], os.environ['I_CC_RIESGOS'] = 'DS RIESGOS', '9810200000'

tags = [
    {'Key': 'I_RESPONSABLE_LT', 'Value': name_ds},
    {'Key': 'I_APLICACION', 'Value': 'SDLF'},
    {'Key': 'I_PROYECTO', 'Value': 'SDLF'},
    {'Key': 'I_AMBIENTE', 'Value': 'DEV'},
    {'Key': 'I_CUENTA', 'Value': account},
    {'Key': 'I_SIGLA', 'Value': 'SAN'},
    {'Key': 'I_TEAM', 'Value': os.environ[f'I_TEAM_{team}']},
    {'Key': 'I_CC', 'Value': os.environ[f'I_CC_{team}']},
]

# Formatear Script

In [26]:
!black --line-length 100 $path_script
!pylint --disable C0103 $path_script

Traceback (most recent call last):
  File "/home/ec2-user/anaconda3/envs/python3/bin/black", line 3, in <module>
    from black import patched_main
  File "src/black/__init__.py", line 33, in <module>
ModuleNotFoundError: No module named 'mypy_extensions'
Traceback (most recent call last):
  File "/home/ec2-user/anaconda3/envs/python3/bin/pylint", line 6, in <module>
    sys.exit(run_pylint())
             ^^^^^^^^^^^^
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pylint/__init__.py", line 33, in run_pylint
    from pylint.lint import Run as PylintRun
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pylint/lint/__init__.py", line 19, in <module>
    from pylint.config.exceptions import ArgumentPreprocessingError
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/pylint/config/__init__.py", line 25, in <module>
    from pylint.config.arguments_provider import UnsupportedAction
  File "/home/ec2-user/anaconda3

# Crear Processor

| Instancia      | CPU Cores | Memoria (GB) |
|----------------|-----------|--------------|  
| ml.m5.large    | 2         | 8            |
| ml.m5.xlarge   | 4         | 16           |
| ml.m5.2xlarge  | 8         | 32           |
| ml.m5.4xlarge  | 16        | 64           |
| ml.m5.12xlarge | 48        | 192          |
| ml.m5.24xlarge | 96        | 384          |

In [27]:
from sagemaker.processing import ScriptProcessor

processor = ScriptProcessor(instance_type='ml.m5.12xlarge',
                            volume_size_in_gb=30,
                            instance_count=1,
                            command=['python3'],
                            image_uri=image_uri,
                            role=role_arn,
                            tags=tags)

# Definir Entradas

In [28]:
from sagemaker.processing import ProcessingInput
nombre_exportado = 'hpo-plaft-pn-masivo-260610-1653-035-1626eaf2'

inputs = []

inputs.append(ProcessingInput(
    input_name='code/utils',
    source='s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PN/MASIVO/DATA_INFERENCIA_PILOTO/Utils/',  # TODO 3: Colocar ubicacion de S3 del script 'utils.py'
    destination='/opt/ml/processing/input/code/utils',
))

inputs.append(ProcessingInput(
    input_name='data',
    source='s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PN/MASIVO/DATA_INFERENCIA_PILOTO/INFERENCIA/periodo=202605/',  # TODO 4: Colocar ubicacion de S3 de los datos del periodo de inferencia
    destination='/opt/ml/processing/input/data',
))

inputs.append(ProcessingInput(
    input_name='models',
    source='s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PN/MASIVO/DATA_INFERENCIA_PILOTO/MODEL/output/hpo-plaft-pn-masivo-260611-2038-045-e0b307d7/output/',  # TODO 5: Colocar ubicacion de S3 de los modelos entrenados
    destination='/opt/ml/processing/input/models',
))

# Definir Salidas

In [29]:
from sagemaker.processing import ProcessingOutput

outputs = []

outputs.append(ProcessingOutput(
    output_name='results',
    source='/opt/ml/processing/output/results',
    destination='s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/MARVIK/MASIVO/REPLICA_OUTPUT',  # TODO 6: Colocar ubicacion de S3 donde van a estar los resultados
))

# Ejecutar Job

In [30]:
model = 'modA_masivo'
table_score = 'scr_modA_agregado_rsk'
partition = '202605'  # TODO 8: Colocar periodo de los datos (YYYYMM), ej: '202605'

arguments = [
    '--model', model,
    '--table-score', table_score,
    '--partition', partition,
]

processor.run(code=path_script,
              inputs=inputs,
              outputs=outputs,
              arguments=arguments)

INFO:sagemaker:Creating processing-job with name sagemaker-python3-2026-06-16-16-06-19-733


.................../usr/local/lib/python3.8/site-packages/dask/dataframe/utils.py:365: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
/usr/local/lib/python3.8/site-packages/dask/dataframe/utils.py:365: FutureWarning: pandas.Float64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
/usr/local/lib/python3.8/site-packages/dask/dataframe/utils.py:365: FutureWarning: pandas.UInt64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
/usr/local/lib/python3.8/site-packages/dask/dataframe/io/parquet/arrow.py:144: FutureWarning:

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:11                                                                                   │
│                                                                                                  │
│    8 │   '--partition', partition,                                                               │
│    9 ]                                                                                           │
│   10                                                                                             │
│ ❱ 11 processor.run(code=path_script,                                                             │
│   12 │   │   │     inputs=inputs,                                                                │
│   13 │   │   │     outputs=outputs,                                                              │
│   14 │   │   │     arguments=arguments)                                                          │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/workflow/pipeline_c │
│ ontext.py:346 in wrapper                                                                         │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/processing.py:688   │
│ in run                                                                                           │
│                                                                                                  │
│    685 │   │   )                                                                                 │
│    686 │   │   self.jobs.append(self.latest_job)                                                 │
│    687 │   │   if wait:                                                                          │
│ ❱  688 │   │   │   self.latest_job.wait(logs=logs)                                               │
│    689 │                                                                                         │
│    690 │   def _include_code_in_inputs(self, inputs, code, kms_key=None):                        │
│    691 │   │   """Converts code to appropriate input and includes in input list.                 │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/processing.py:1113  │
│ in wait                                                                                          │
│                                                                                                  │
│   1110 │   │                                                                                     │
│   1111 │   │   """                                                                               │
│   1112 │   │   if logs:                                                                          │
│ ❱ 1113 │   │   │   self.sagemaker_session.logs_for_processi

## Bibliotecas Instaladas

- `image_uri = f'{account}.dkr.ecr.us-east-1.amazonaws.com/sagemaker-python3:3.8.15-cpu'`  

    ```
    catboost           1.1
    dask               2.11.0
    joblib             1.2.0
    lightgbm           3.3.3
    numpy              1.23.5
    pandas             1.5.1
    pyarrow            10.0.0
    scikit-learn       1.1.2
    scipy              1.9.3
    xgboost            1.6.2
    ```